In [226]:
import numpy as np
import pandas as pd
import plotly.express as px
import tqdm

In [227]:
np.random.seed(0)

In [228]:
threshold_min, threshold_max, threshold_delta = 0., 1., 0.1

In [229]:
def bayesian_update(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [ ]:
def manipulation_thresholds(thresholds, priors, c):
    if np.sum(priors) == 0:
        return thresholds, priors, thresholds

    thresholds_m = np.maximum(0, thresholds - (bayesian_update(priors) / c))
    if len(thresholds_m) <= 1:
        return thresholds_m, priors, thresholds
    
    priors_updated = []
    thresholds_updated = []
    manip_thresholds_updated = []
    manip_thresh_prev = -np.inf
    for i, manip_thresh in enumerate(thresholds_m):
        if manip_thresh <= manip_thresh_prev:
            del priors_updated[i-1]
            del thresholds_updated[i-1]
            del manip_thresholds_updated[i-1]
            priors_updated.append(priors[i-1] + priors[i])
        else:
            priors_updated.append(priors[i])
        thresholds_updated.append(thresholds[i])
        manip_thresholds_updated.append(thresholds_m[i])
        manip_thresh_prev = manip_thresh

    return manip_thresholds_updated, priors_updated, thresholds_updated

In [231]:
def balance_priors(priors, random=False):
    total = np.sum(priors)
    if total == 1:
        return priors
    indices = priors == 0
    remainder = 1 - total
    if random:
        p = np.random.rand(indices.sum())
        p = (p / p.sum()) * remainder
    else:
        p = remainder / indices.sum()
    priors[indices] = p
    return priors

In [232]:
def accuracy_loss(thresholds, x_manipulation, threshold_true):
    losses = []
    for i in range(len(thresholds)):
        if thresholds[i] < threshold_true:
            loss = (threshold_true - x_manipulation[i]) / (threshold_max - threshold_min)
        else:
            loss = np.abs(x_manipulation[i] - threshold_true) / (threshold_max - threshold_min)
        losses.append(loss)
    return np.array(losses)

In [233]:
def best_response(x, thresholds, priors, c):
    posteriors = bayesian_update(priors)
    
    mask = thresholds > x
    search_space = [x]
    search_space.extend(thresholds[mask].tolist())
    utilities = []
    for i, x_p in enumerate(search_space):
        utility = 0.
        for j in range(len(thresholds)):
            utility += posteriors[j] * (x_p >= thresholds[j])
        cost = c*abs(x_p - x)
        util = utility - cost
        utilities.append(util)
    
    return search_space[np.argmax(utilities)]

In [234]:
thresholds = np.arange(threshold_min+threshold_delta, threshold_max, threshold_delta).round(4)

priors = np.zeros_like(thresholds)
priors[0] = 0.
priors[1] = 0.
priors[2] = 0.
priors[3] = 0.
priors[4] = 0.
priors[5] = 0.

balance_priors(priors[6:], random=True)
print(np.sum(priors))
# assert np.sum(priors) == 1

pd.DataFrame({"threshold": thresholds, "priors": priors}).round(3).T

0.9999999999999998


,0,1,2,3,4,5,6,7,8
threshold,0.1,0.2,0.3,0.4,0.5,0.6,0.700,0.800,0.900
priors,0.0,0.0,0.0,0.0,0.0,0.0,0.294,0.383,0.323


In [235]:
c = 5
threshold_true = 0.5
n = len(thresholds)
partitions = [i for i in range(n)]

In [236]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        # Option 1: put `first` in each existing block
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        # Option 2: put `first` in its own new block
        yield [[first]] + smaller

In [237]:
parts = set_partitions(partitions)
partitions_set = []
for part in parts:
    partitions_set.append(part)

In [238]:
# partitions_set
# partitions_set = [[[0,1,2], [3,4,5], [6,7,8]]]

In [ ]:
results = {
    "threshold": [],
    "prior": [],
    "partition": [],
    "accuracy_loss": [],
    "partition_loss": [],
    "manip_threshold": [],
}

for i, partition in enumerate(partitions):
    threshold_p = thresholds[partition]
    priors_p = priors[partition]
    x_manipulation_p, priors_p, threshold_p = manipulation_thresholds(threshold_p, priors_p, c)
    acc_loss = accuracy_loss(threshold_p, x_manipulation_p, threshold_true)
    partition_loss = np.dot(acc_loss, bayesian_update(priors_p))
    
    for ti, t in enumerate(threshold_p):
        results["threshold"].append(t)
        results["prior"].append(priors_p[ti])
        results["partition"].append(f"{i}")
        results["accuracy_loss"].append(acc_loss[ti])
        results["partition_loss"].append(partition_loss)
        results["manip_threshold"].append(x_manipulation_p[ti])

accuracy_loss =np.dot(df.T["accuracy_loss"].values, df.T["prior"].values)

100%|██████████| 21147/21147 [00:00<00:00, 28274.40it/s]
